# Notebook 9: Grouping & Aggregation
**Filename:** `09_GroupBy.ipynb`  
**Topics Covered:** `groupby()`, `agg()`, `transform()`, `filter()`, Pivot Tables, Crosstab

---

## 1. Grouping Data (`groupby`)

### Concept Explanation
`groupby()` implements the "split-apply-combine" pattern. It splits data into groups based on categorical keys, applies an aggregation or computation function, and combines the results into a single object.

### Real-world Example
Grouping regional weather station data by month to calculate average monthly temperature.

### Business Example
Grouping sales transactions by region and store department to compute total sales revenue.

### AI/ML Example
Grouping user interaction logs by `User_ID` to aggregate total activity duration before creating model feature sets.

In [1]:
import pandas as pd

# Sample Dataset
df = pd.DataFrame({
    'Store': ['North', 'North', 'South', 'South', 'East', 'East'],
    'Category': ['Tech', 'Furniture', 'Tech', 'Furniture', 'Tech', 'Furniture'],
    'Sales': [1200, 800, 1500, 400, 900, 600],
    'Quantity': [10, 5, 12, 3, 8, 4]
})

# Basic GroupBy: Average Sales by Store
sales_by_store = df.groupby('Store')['Sales'].mean()

# GroupBy across multiple columns
sales_multi = df.groupby(['Store', 'Category'])['Sales'].sum()

print("Average Sales by Store:\n", sales_by_store)
print("\nTotal Sales by Store & Category:\n", sales_multi)

Average Sales by Store:
 Store
East      750.0
North    1000.0
South     950.0
Name: Sales, dtype: float64

Total Sales by Store & Category:
 Store  Category 
East   Furniture     600
       Tech          900
North  Furniture     800
       Tech         1200
South  Furniture     400
       Tech         1500
Name: Sales, dtype: int64


---

## 2. Multi-Metric & Custom Aggregations (`agg`)

### Concept Explanation
`agg()` (or `aggregate()`) allows applying multiple aggregation functions simultaneously across columns or applying custom column-specific aggregation mappings using dictionaries.

### Real-world Example
Calculating min, max, and median values of air quality indices per city.

### Business Example
Computing total revenue, average order value, and distinct customer counts per store location in a single call.

### AI/ML Example
Aggregating transaction history to create summary features like mean spend and total transaction counts.

In [2]:
import pandas as pd

# Multiple aggregations on a single column
sales_summary = df.groupby('Category')['Sales'].agg(['sum', 'mean', 'max'])

# Column-specific aggregations
custom_agg = df.groupby('Store').agg({
    'Sales': ['sum', 'mean'],
    'Quantity': 'max'
})

print("Category Sales Summary:\n", sales_summary)
print("\nCustom Column Aggregations:\n", custom_agg)

Category Sales Summary:
             sum    mean   max
Category                     
Furniture  1800   600.0   800
Tech       3600  1200.0  1500

Custom Column Aggregations:
       Sales         Quantity
        sum    mean      max
Store                       
East   1500   750.0        8
North  2000  1000.0       10
South  1900   950.0       12


---

## 3. Group Transformation (`transform`)

### Concept Explanation
`transform()` calculates group-level statistics but returns a Series with the **same index and shape** as the original DataFrame. It broadcasts group values back to individual rows.

### Real-world Example
Calculating each day's temperature relative to the monthly average temperature.

### Business Example
Calculating each transaction's percentage contribution relative to its regional sales total.

### AI/ML Example
Standardizing/normalizing numerical features within dynamic subgroups (e.g., group z-score normalization).

In [3]:
import pandas as pd

# Calculate store total sales and broadcast back to original shape
df['Store_Total_Sales'] = df.groupby('Store')['Sales'].transform('sum')

# Calculate percentage contribution of each transaction
df['Pct_Of_Store_Sales'] = (df['Sales'] / df['Store_Total_Sales']) * 100

print("DataFrame with Transformed Features:\n", df[['Store', 'Category', 'Sales', 'Store_Total_Sales', 'Pct_Of_Store_Sales']])

DataFrame with Transformed Features:
    Store   Category  Sales  Store_Total_Sales  Pct_Of_Store_Sales
0  North       Tech   1200               2000           60.000000
1  North  Furniture    800               2000           40.000000
2  South       Tech   1500               1900           78.947368
3  South  Furniture    400               1900           21.052632
4   East       Tech    900               1500           60.000000
5   East  Furniture    600               1500           40.000000


---

## 4. Group Filtering (`filter`)

### Concept Explanation
`filter()` subsets groups based on group-level boolean conditions. Groups that do not satisfy the aggregate condition are dropped entirely from the output.

### Real-world Example
Retaining weather station logs only for stations that recorded more than 100 observations.

### Business Example
Filtering out small stores where total cumulative revenue across all sales is below $2,000.

### AI/ML Example
Removing user cohorts from training datasets if they have fewer than 5 historical interaction events.

In [4]:
import pandas as pd

# Keep only stores whose total cumulative sales exceed $1500
high_volume_stores = df.groupby('Store').filter(lambda g: g['Sales'].sum() > 1500)

print("Filtered DataFrame (Stores with sum(Sales) > 1500):\n", high_volume_stores)

Filtered DataFrame (Stores with sum(Sales) > 1500):
    Store   Category  Sales  Quantity  Store_Total_Sales  Pct_Of_Store_Sales
0  North       Tech   1200        10               2000           60.000000
1  North  Furniture    800         5               2000           40.000000
2  South       Tech   1500        12               1900           78.947368
3  South  Furniture    400         3               1900           21.052632


---

## 5. Pivot Tables (`pivot_table`)

### Concept Explanation
`pivot_table()` reshapes data into a multidimensional grid. It takes row keys (`index`), column keys (`columns`), values to aggregate (`values`), and aggregate functions (`aggfunc`).

### Real-world Example
Creating a 2D weather matrix showing average temperatures across months (rows) and cities (columns).

### Business Example
Generating an executive grid of total sales revenue split by Region (rows) and Product Category (columns), with row/column subtotals (`margins=True`).

### AI/ML Example
Creating tabular summary heatmaps of model evaluation metrics across hyperparameter combinations.

In [5]:
import pandas as pd

# Create Pivot Table with row and column dimensions
pivot = pd.pivot_table(
    df, 
    values='Sales', 
    index='Store', 
    columns='Category', 
    aggfunc='sum', 
    fill_value=0,
    margins=True  # Adds subtotal row/column
)

print("Sales Pivot Table:\n", pivot)

Sales Pivot Table:
 Category  Furniture  Tech   All
Store                          
East            600   900  1500
North           800  1200  2000
South           400  1500  1900
All            1800  3600  5400


---

## 6. Cross-Tabulation (`crosstab`)

### Concept Explanation
`pd.crosstab()` computes frequency tables (counts or normalized percentages) across two or more categorical factors.

### Real-world Example
Tabulating the count of sunny vs. rainy days observed across different geographic zones.

### Business Example
Analyzing the distribution frequency of payment methods (UPI, Credit, Cash) across different customer age tiers.

### AI/ML Example
Generating confusion matrices comparing actual target class labels against predicted class labels.

In [6]:
import pandas as pd

# Frequency cross-tabulation of Store vs Category
ct_counts = pd.crosstab(df['Store'], df['Category'])

# Normalized cross-tabulation (Row Percentages)
ct_props = pd.crosstab(df['Store'], df['Category'], normalize='index') * 100

print("Frequency Counts:\n", ct_counts)
print("\nRow Percentages (%):\n", ct_props)

Frequency Counts:
 Category  Furniture  Tech
Store                    
East              1     1
North             1     1
South             1     1

Row Percentages (%):
 Category  Furniture  Tech
Store                    
East           50.0  50.0
North          50.0  50.0
South          50.0  50.0


---

## Minimum 5 Interview Questions with Answers

1. **What is the difference between `groupby().agg()` and `groupby().transform()`?**  
   * **Answer:** `agg()` reduces each group down to a single aggregate scalar value, returning a DataFrame indexed by the group keys. `transform()` applies a group calculation and returns an object with the same index and row count as the original DataFrame.

2. **How does `groupby().filter()` work in Pandas?**  
   * **Answer:** `groupby().filter()` evaluates a boolean function across an entire group data slice. If the function evaluates to `True` for that group, all rows belonging to that group are retained; otherwise, the whole group is dropped.

3. **What is the difference between `pivot_table()` and `groupby()`?**  
   * **Answer:** `groupby()` returns data in long format (often with a MultiIndex). `pivot_table()` reshapes data into wide multi-dimensional grids (rows vs columns matrix) with built-in parameter options for handling missing values (`fill_value`) and subtotals (`margins`).

4. **When would you use `pd.crosstab()` over `pivot_table()`?**  
   * **Answer:** `pd.crosstab()` is specifically optimized for quick categorical frequency tables and cross-tabulation percentages from raw Series, whereas `pivot_table()` requires an existing DataFrame and can aggregate continuous numerical data using varied statistical functions.

5. **How can you aggregate different statistical operations on different columns within a single `groupby()` statement?**  
   * **Answer:** By passing a dictionary mapping to `agg()`, e.g., `df.groupby('Key').agg({'Sales': 'sum', 'Quantity': 'mean'})`.

---

## Self Reflection
* **What I Learned:** Mastered split-apply-combine logic via `groupby()`, multi-metric aggregation (`agg()`), group feature engineering (`transform()`), group filtering (`filter()`), wide-format pivoting (`pivot_table()`), and categorical frequency table generation (`crosstab()`).
* **Key Takeaway:** Grouped aggregations and pivot transformations turn granular row-level data into high-level business intelligence metrics and group-level machine learning features.
